# LeetCode 1814 — Count Nice Pairs in an Array bằng Python

Link đề: https://leetcode.com/problems/count-nice-pairs-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Counting

Skeleton LeetCode:

```python
class Solution:
    def countNicePairs(self, nums: List[int]) -> int:
        ...
```

Lộ trình: **duyệt mọi cặp** `O(n²)` → **biến đổi đại số** `a + rev(b) = b + rev(a)` ⇔ `a − rev(a) = b − rev(b)` → **sắp xếp khóa rồi đếm nhóm** `O(n log n)` → **`Counter` + tổ hợp `C(v, 2)`** → **đếm một lượt "cộng dồn trước khi tăng"** `O(n · d)` (lời giải nộp).

## 1. Mục tiêu học

1. Cài đặt `rev(x)` (đảo chữ số) bằng **số học** (`% 10`, `// 10`) và bằng **chuỗi**, hiểu cách xử lý số 0 ở cuối (`rev(120) = 21`).
2. **Biến đổi phương trình** để tách biến: chuyển mọi thứ của `i` sang một vế, của `j` sang vế kia ⇒ điều kiện thành "**cùng khóa**".
3. Đếm số cặp `i < j` có cùng khóa bằng **bảng băm**: hoặc `Σ C(v, 2)`, hoặc "mỗi phần tử mới ghép với mọi phần tử cũ cùng khóa".
4. Áp dụng **modulo `10⁹ + 7`** đúng chỗ.
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed; tránh lỗi đếm cặp có thứ tự / đếm cả `(i, i)`.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm các số nguyên **không âm**. Định nghĩa `rev(x)` là số đảo ngược chữ số của `x`; ví dụ `rev(123) = 321`, `rev(120) = 21`. Cặp chỉ số `(i, j)` là **đẹp** (nice) nếu:
- `0 ≤ i < j < nums.length`
- `nums[i] + rev(nums[j]) == nums[j] + rev(nums[i])`

Trả về **số cặp đẹp**, modulo `10⁹ + 7`.

**Ví dụ 1:** `nums = [42,11,1,97]` → `2`: cặp `(0,3)`: `42 + 79 = 121 = 97 + 24`; cặp `(1,2)`: `11 + 1 = 12 = 1 + 11`.

**Ví dụ 2:** `nums = [13,10,35,24,76]` → `4`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁹`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from typing import List

MOD = 10**9 + 7
examples = [([42, 11, 1, 97], 2), ([13, 10, 35, 24, 76], 4)]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[42, 11, 1, 97] -> kỳ vọng 2
[13, 10, 35, 24, 76] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Đảo chữ số bằng số học

Lặp: lấy chữ số cuối `x % 10`, đẩy vào `y = y * 10 + d`, bỏ chữ số cuối `x //= 10`. Số 0 ở cuối `x` trở thành số 0 ở **đầu** `y` — tự biến mất.

In [2]:
def rev(x: int) -> int:
    y = 0
    while x:
        y = y * 10 + x % 10
        x //= 10
    return y

print([rev(v) for v in [123, 120, 0, 7, 1000, 1000000000]])

[321, 21, 0, 7, 1, 1]


### 3.2 Đảo chữ số bằng chuỗi

`int(str(x)[::-1])` — `int("021") == 21` nên số 0 đầu cũng tự mất. Ngắn gọn, tốc độ tương đương (thậm chí nhanh hơn trong CPython vì chạy bằng C).

In [3]:
print([int(str(v)[::-1]) for v in [123, 120, 0, 7, 1000]])
assert all(rev(v) == int(str(v)[::-1]) for v in range(100000))
print("hai cách cho cùng kết quả trên 0..99999")

[321, 21, 0, 7, 1]
hai cách cho cùng kết quả trên 0..99999


### 3.3 `Counter` và tổ hợp chập 2

Có `v` phần tử cùng nhóm thì số cặp không thứ tự là `C(v, 2) = v(v−1)/2`. `Counter(iterable)` đếm tần suất.

In [4]:
c = Counter(["a", "b", "a", "a", "c", "b"])
print(c, "| số cặp:", {k: v * (v - 1) // 2 for k, v in c.items()})

Counter({'a': 3, 'b': 2, 'c': 1}) | số cặp: {'a': 3, 'b': 1, 'c': 0}


### 3.4 Mẫu "đếm cặp một lượt"

Duyệt từ trái sang phải; khi gặp phần tử có khóa `k`, nó tạo cặp với **mọi** phần tử trước đó cùng khóa: `ans += seen[k]`, rồi `seen[k] += 1`. Tổng cuối bằng `Σ C(v, 2)` (0 + 1 + … + (v−1)).

In [5]:
seen, ans = defaultdict(int), 0
for k in ["a", "b", "a", "a"]:
    ans += seen[k]
    seen[k] += 1
print("số cặp:", ans, "| C(3,2) + C(1,2) =", 3 + 0)

số cặp: 3 | C(3,2) + C(1,2) = 3


### 3.5 Modulo `10⁹ + 7`

Số cặp tối đa `C(10⁵, 2) ≈ 5·10⁹` — vượt `10⁹ + 7` nên phải lấy dư. Python không tràn số nên có thể lấy dư **một lần** ở cuối; ở C++/Java cần `long` và lấy dư thường xuyên.

In [6]:
n = 10**5
print(n * (n - 1) // 2, "->", n * (n - 1) // 2 % MOD)

4999950000 -> 999949972


## 4. Ý tưởng nền tảng / trực giác

Điều kiện `a + rev(b) = b + rev(a)` có **hai biến lẫn lộn** ở hai vế. Chuyển vế:

`a − rev(a) = b − rev(b)`.

Giờ mỗi vế chỉ phụ thuộc **một** phần tử! Đặt `key(x) = x − rev(x)`. Cặp `(i, j)` đẹp ⇔ `key(nums[i]) == key(nums[j])`.

Tính tay Ví dụ 2 `[13, 10, 35, 24, 76]`:

| x | rev(x) | key = x − rev(x) |
|---|---|---|
| 13 | 31 | −18 |
| 10 | 1 | 9 |
| 35 | 53 | −18 |
| 24 | 42 | −18 |
| 76 | 67 | 9 |

Nhóm `−18`: 3 phần tử ⇒ `C(3,2) = 3` cặp; nhóm `9`: 2 phần tử ⇒ 1 cặp. Tổng **4**. ✔

In [7]:
nums = [13, 10, 35, 24, 76]
for x in nums:
    print(f"x={x:>3} rev={rev(x):>3} key={x - rev(x):>4}")
print(Counter(x - rev(x) for x in nums))

x= 13 rev= 31 key= -18
x= 10 rev=  1 key=   9
x= 35 rev= 53 key= -18
x= 24 rev= 42 key= -18
x= 76 rev= 67 key=   9
Counter({-18: 3, 9: 2})


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp

In [8]:
def nice_pairs_brute(nums):
    n = len(nums)
    r = [rev(x) for x in nums]          # tính rev trước cho nhẹ
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + r[j] == nums[j] + r[i]:
                cnt += 1
    return cnt % MOD

for nums, want in examples:
    print(nums, "->", nice_pairs_brute(nums), "| kỳ vọng", want)

[42, 11, 1, 97] -> 2 | kỳ vọng 2
[13, 10, 35, 24, 76] -> 4 | kỳ vọng 4


**Độ phức tạp:** `n(n−1)/2` cặp ⇒ `O(n²)`; với `n = 10⁵` là `5·10⁹` phép so — quá chậm. Nút thắt: kiểm từng cặp, trong khi điều kiện thực chất chỉ là "cùng nhóm".

In [9]:
rng = random.Random(0)
arr = [rng.randint(0, 10**9) for _ in range(3000)]
t = time.perf_counter(); nice_pairs_brute(arr); dt = time.perf_counter() - t
print(f"n=3000: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 3000) ** 2:.0f}s")

n=3000: 0.31s -> n=10^5 ước tính ≈ 348s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tách biến):** `nums[i] + rev(nums[j]) = nums[j] + rev(nums[i])` ⇔ `nums[i] − rev(nums[i]) = nums[j] − rev(nums[j])`. Phép chuyển vế là **tương đương** (cộng/trừ cùng một số hai vế), nên không mất/không thừa cặp nào.

**Quy tắc:** *nếu* hai phần tử có cùng `key` *thì* chúng tạo cặp đẹp; *nếu* khác `key` *thì* không. Bài toán thành "**đếm số cặp `i < j` cùng nhóm**" — kinh điển, giải bằng bảng băm.

**Invariant (bản một lượt):** sau khi xử lý `nums[0..t]`, `seen[k]` = số phần tử trong đoạn đó có khóa `k`, và `ans` = số cặp đẹp `(i, j)` với `j ≤ t`.

Lưu ý `key` có thể **âm** (`13 − 31 = −18`) — không sao với `dict`. (Đừng nhầm sang `rev(x) − x`: cũng đúng nếu dùng nhất quán, vì chỉ đổi dấu mọi khóa.)

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp khóa, đếm các khúc bằng nhau (`O(n log n)`)

Không cần bảng băm: sắp xếp danh sách khóa, các khóa bằng nhau đứng liền nhau thành **khúc** dài `v` ⇒ cộng `C(v, 2)`.

In [10]:
def nice_pairs_sort(nums):
    keys = sorted(x - rev(x) for x in nums)
    ans, run = 0, 1
    for i in range(1, len(keys) + 1):
        if i < len(keys) and keys[i] == keys[i - 1]:
            run += 1
        else:
            ans += run * (run - 1) // 2
            run = 1
    return ans % MOD

for nums, want in examples:
    print(nice_pairs_sort(nums), want)

2 2
4 4


**Độ phức tạp:** `O(n · d)` tính khóa (`d ≤ 10` chữ số) + `O(n log n)` sắp xếp; bộ nhớ `O(n)`.

### 7.2 Cách 3 — `Counter` + `Σ C(v, 2)` (`O(n · d)`)

In [11]:
class SolutionCounter:
    def countNicePairs(self, nums: List[int]) -> int:
        cnt = Counter(x - rev(x) for x in nums)
        return sum(v * (v - 1) // 2 for v in cnt.values()) % MOD

for nums, want in examples:
    print(SolutionCounter().countNicePairs(nums), want)

2 2
4 4


### 7.3 Cách 4 — Một lượt "cộng dồn trước khi tăng"

Không cần công thức tổ hợp; mỗi phần tử mới ghép với mọi phần tử cũ cùng khóa. Kiểu viết này mở rộng tốt cho biến thể (ví dụ chỉ đếm cặp có `j − i ≤ K`, hay cặp thỏa điều kiện thêm).

In [12]:
def nice_pairs_one_pass_demo(nums):
    seen, ans = defaultdict(int), 0
    for j, x in enumerate(nums):
        k = x - int(str(x)[::-1])
        print(f"j={j} x={x:>3} key={k:>4}: ghép với {seen[k]} phần tử trước cùng khóa -> ans={ans + seen[k]}")
        ans += seen[k]
        seen[k] += 1
    return ans % MOD

nice_pairs_one_pass_demo([13, 10, 35, 24, 76])

j=0 x= 13 key= -18: ghép với 0 phần tử trước cùng khóa -> ans=0
j=1 x= 10 key=   9: ghép với 0 phần tử trước cùng khóa -> ans=0
j=2 x= 35 key= -18: ghép với 1 phần tử trước cùng khóa -> ans=1
j=3 x= 24 key= -18: ghép với 2 phần tử trước cùng khóa -> ans=3
j=4 x= 76 key=   9: ghép với 1 phần tử trước cùng khóa -> ans=4


4

In [13]:
class SolutionOnePass:
    def countNicePairs(self, nums: List[int]) -> int:
        seen, ans = defaultdict(int), 0
        for x in nums:
            k = x - int(str(x)[::-1])
            ans += seen[k]
            seen[k] += 1
        return ans % MOD

for nums, want in examples:
    print(SolutionOnePass().countNicePairs(nums), want)

2 2
4 4


**Độ phức tạp:** `O(n · d)` thời gian, `O(n)` bộ nhớ (số khóa khác nhau ≤ `n`).

### 7.4 Ghi chú hiệu năng: `rev` bằng số học hay bằng chuỗi?

Cả hai đều `O(d)`. Trong CPython, `int(str(x)[::-1])` thường **nhanh hơn** vòng `while` vì phần việc chạy trong mã C; còn ở ngôn ngữ biên dịch (C++/Java) thì cách số học nhanh hơn. Đo thử:

In [14]:
vals = [random.Random(3).randint(0, 10**9) for _ in range(200000)]
t = time.perf_counter(); a1 = [rev(v) for v in vals]; t1 = time.perf_counter() - t
t = time.perf_counter(); a2 = [int(str(v)[::-1]) for v in vals]; t2 = time.perf_counter() - t
assert a1 == a2
print(f"số học: {1000*t1:.0f} ms | chuỗi: {1000*t2:.0f} ms (2·10^5 số)")

số học: 98 ms | chuỗi: 49 ms (2·10^5 số)


## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 `[42, 11, 1, 97]`:

| j | x | rev | key | `seen[key]` trước | `ans` sau | `seen` sau |
|---|---|---|---|---|---|---|
| 0 | 42 | 24 | 18 | 0 | 0 | {18:1} |
| 1 | 11 | 11 | 0 | 0 | 0 | {18:1, 0:1} |
| 2 | 1 | 1 | 0 | 1 | 1 | {18:1, 0:2} |
| 3 | 97 | 79 | 18 | 1 | **2** | {18:2, 0:2} |

Cặp tìm được: `(1,2)` khi `j=2`, `(0,3)` khi `j=3`. ✔

In [15]:
def trace_pairs(nums):
    seen = defaultdict(list)          # lưu luôn chỉ số để in cặp
    ans = 0
    for j, x in enumerate(nums):
        k = x - rev(x)
        pairs = [(i, j) for i in seen[k]]
        ans += len(pairs)
        print(f"j={j} x={x:>3} rev={rev(x):>3} key={k:>4} | cặp mới {pairs} | ans={ans}")
        seen[k].append(j)
    return ans

trace_pairs([42, 11, 1, 97])

j=0 x= 42 rev= 24 key=  18 | cặp mới [] | ans=0
j=1 x= 11 rev= 11 key=   0 | cặp mới [] | ans=0
j=2 x=  1 rev=  1 key=   0 | cặp mới [(1, 2)] | ans=1
j=3 x= 97 rev= 79 key=  18 | cặp mới [(0, 3)] | ans=2


2

In [16]:
trace_pairs([13, 10, 35, 24, 76])

j=0 x= 13 rev= 31 key= -18 | cặp mới [] | ans=0
j=1 x= 10 rev=  1 key=   9 | cặp mới [] | ans=0
j=2 x= 35 rev= 53 key= -18 | cặp mới [(0, 2)] | ans=1
j=3 x= 24 rev= 42 key= -18 | cặp mới [(0, 3), (2, 3)] | ans=3
j=4 x= 76 rev= 67 key=   9 | cặp mới [(1, 4)] | ans=4


4

## 9. Code nộp LeetCode

In [17]:
from collections import defaultdict
from typing import List

class Solution:
    def countNicePairs(self, nums: List[int]) -> int:
        def rev(x: int) -> int:
            y = 0
            while x:
                y = y * 10 + x % 10
                x //= 10
            return y

        seen = defaultdict(int)          # key = x - rev(x) -> số lần đã gặp
        ans = 0
        for x in nums:
            k = x - rev(x)
            ans += seen[k]               # ghép với mọi phần tử trước cùng khóa
            seen[k] += 1
        return ans % (10**9 + 7)

for nums, want in examples:
    assert Solution().countNicePairs(nums) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (tương đương):** với mọi số nguyên `a, b`: `a + rev(b) = b + rev(a)` ⇔ `a − rev(a) = b − rev(b)` — trừ `rev(a) + rev(b)` hai vế (phép biến đổi khả nghịch). Do đó tập cặp đẹp = tập cặp `i < j` có `key(nums[i]) = key(nums[j])`.

**Đếm đúng (Cách 4):** mỗi cặp đẹp `(i, j)` với `i < j` được đếm **đúng một lần**: ở bước `j`, `seen[key]` đã chứa `i` (vì `i < j` đã xử lý), và không được đếm ở bước nào khác (ở bước `i`, `j` chưa xuất hiện). Không đếm `(j, j)` vì `seen[k]` được tăng **sau** khi cộng.

**Cách 3:** nhóm có `v` phần tử đóng góp đúng `C(v, 2)` cặp không thứ tự `{i, j}`, mỗi cặp ứng với đúng một cặp có thứ tự `i < j`. Các nhóm rời nhau ⇒ cộng lại.

**Modulo:** lấy dư một lần ở cuối cho kết quả bằng lấy dư dần (tính chất `(a + b) mod p`).

Kiểm chứng Bổ đề bằng máy trên nhiều cặp ngẫu nhiên — cả hai chiều của "⇔":

In [18]:
rng = random.Random(10)
both = 0
for _ in range(200000):
    a, b = rng.randint(0, 300), rng.randint(0, 300)
    lhs = a + rev(b) == b + rev(a)
    rhs = a - rev(a) == b - rev(b)
    assert lhs == rhs
    both += lhs
print("200000 cặp: điều kiện gốc ⇔ cùng khóa; số cặp đẹp gặp được:", both)

200000 cặp: điều kiện gốc ⇔ cùng khóa; số cặp đẹp gặp được: 11785


## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `d ≤ 10` = số chữ số tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n² + n·d)` | `O(n)` |
| 2. Sắp xếp khóa | `O(n·d + n log n)` | `O(n)` |
| 3. Counter + `C(v,2)` | `O(n·d)` trung bình | `O(n)` |
| 4. Một lượt | `O(n·d)` trung bình | `O(n)` |

"Trung bình" vì thao tác `dict` `O(1)` trung bình. Đo thực nghiệm ở `n = 10⁵`:

In [19]:
rng = random.Random(1814)
big = [rng.randint(0, 10**9) for _ in range(10**5)]
pal = [int(str(v) + str(v)[-2::-1]) for v in range(1, 10**5 + 1)]                 # số đối xứng (≤ 9 chữ số): key = 0
for label, data in [("ngẫu nhiên", big), ("nhiều số đối xứng", pal)]:
    res = []
    for name, f in [("Cách 2", nice_pairs_sort), ("Cách 3", SolutionCounter().countNicePairs),
                    ("Cách 4", SolutionOnePass().countNicePairs), ("Nộp", Solution().countNicePairs)]:
        t = time.perf_counter(); r = f(data); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1

[ngẫu nhiên] Cách 2: 86673 trong 66 ms
[ngẫu nhiên] Cách 3: 86673 trong 51 ms
[ngẫu nhiên] Cách 4: 86673 trong 29 ms


[ngẫu nhiên] Nộp: 86673 trong 56 ms
[nhiều số đối xứng] Cách 2: 999949972 trong 47 ms


[nhiều số đối xứng] Cách 3: 999949972 trong 43 ms
[nhiều số đối xứng] Cách 4: 999949972 trong 18 ms
[nhiều số đối xứng] Nộp: 999949972 trong 45 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
sols = [Solution(), SolutionCounter(), SolutionOnePass()]
edge = examples + [
    ([5], 0),                         # một phần tử
    ([0, 0], 1),                      # 0 - rev(0) = 0
    ([10, 1], 0),                     # key(10) = 9, key(1) = 0 -> khác nhóm
    ([1, 2, 3, 11, 22], 10),          # mọi số có key 0 -> C(5,2)
    ([10**9, 1], 0),
    ([120, 21], 0),
]
for nums, want in edge:
    assert nice_pairs_brute(nums) == want
    for s in sols:
        assert s.countNicePairs(nums) == want, (nums, s)
    assert nice_pairs_sort(nums) == want
    print("OK", nums, "->", want)

OK [42, 11, 1, 97] -> 2
OK [13, 10, 35, 24, 76] -> 4
OK [5] -> 0
OK [0, 0] -> 1
OK [10, 1] -> 0
OK [1, 2, 3, 11, 22] -> 10
OK [1000000000, 1] -> 0
OK [120, 21] -> 0


Chú ý case `[10, 1]`: trực giác "10 đảo thành 01 = 1 nên giống 1" là sai — `key(10) = 10 − 1 = 9`, `key(1) = 0`, khác nhóm ⇒ 0 cặp. Mọi kỳ vọng tự dựng ở đây (không có trong đề gốc) đều được `assert` lại bằng vét cạn.

### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

Dùng hai kiểu dữ liệu: giá trị nhỏ (nhiều khóa trùng) và giá trị lớn tới `10⁹`.

In [21]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 30)
    hi = rng.choice([20, 120, 1000, 10**9])
    nums = [rng.randint(0, hi) for _ in range(n)]
    want = nice_pairs_brute(nums)
    for s in sols:
        assert s.countNicePairs(nums) == want, nums
    assert nice_pairs_sort(nums) == want
print("1500 case ngẫu nhiên khớp vét cạn")

1500 case ngẫu nhiên khớp vét cạn


### 12.3 Kiểm tra modulo trên dữ liệu cực đoan

`10⁵` số cùng khóa (ví dụ toàn số 7) ⇒ `C(10⁵, 2) = 4999950000` vượt `10⁹ + 7`.

In [22]:
allsame = [7] * 10**5
want = (10**5 * (10**5 - 1) // 2) % MOD
for s in sols:
    assert s.countNicePairs(allsame) == want
print("C(10^5, 2) mod p =", want)

C(10^5, 2) mod p = 999949972


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Nhóm theo `x + rev(x)` (chuyển vế sai)

In [23]:
def wrong_sum_key(nums):
    cnt = Counter(x + rev(x) for x in nums)              # SAI: không tách được biến
    return sum(v * (v - 1) // 2 for v in cnt.values()) % MOD

for nums, want in examples:
    print(nums, "sai:", wrong_sum_key(nums), "| đúng:", want)

[42, 11, 1, 97] sai: 0 | đúng: 2
[13, 10, 35, 24, 76] sai: 0 | đúng: 4


**Vì sao sai:** `a + rev(b) = b + rev(a)` không tương đương `a + rev(a) = b + rev(b)`. Ví dụ `42, 97`: `42 + 24 = 66 ≠ 97 + 79 = 176` nhưng là cặp đẹp. Chuyển vế đúng: **trừ** `rev(a)` và `rev(b)` hai vế ⇒ `a − rev(a) = b − rev(b)`.

### Lỗi 2 — Đếm `v²` hoặc `v(v+1)/2` thay vì `v(v−1)/2`

In [24]:
def wrong_count_formula(nums):
    cnt = Counter(x - rev(x) for x in nums)
    return (sum(v * v for v in cnt.values()) // 2) % MOD   # SAI: tính cả (i, i)

for nums, want in examples:
    print(nums, "sai:", wrong_count_formula(nums), "| đúng:", want)

[42, 11, 1, 97] sai: 4 | đúng: 2
[13, 10, 35, 24, 76] sai: 6 | đúng: 4


**Vì sao sai:** `v²` đếm mọi cặp **có thứ tự kể cả `(i, i)`**; chia 2 vẫn còn thừa `v/2` cặp `(i, i)`. Số cặp `i < j` là `v(v−1)/2`. Tương tự, ở bản một lượt, nếu tăng `seen[k]` **trước** khi cộng thì mỗi phần tử tự ghép với chính nó.

### Lỗi 3 — Quên modulo

In [25]:
def wrong_no_mod(nums):
    cnt = Counter(x - rev(x) for x in nums)
    return sum(v * (v - 1) // 2 for v in cnt.values())     # SAI: thiếu % MOD

print("sai:", wrong_no_mod(allsame), "| đúng:", Solution().countNicePairs(allsame))

sai: 4999950000 | đúng: 999949972


**Vì sao sai:** đề yêu cầu trả kết quả modulo `10⁹ + 7`; Python không tràn số nên lỗi này **không** gây crash mà chỉ ra đáp án sai trên test lớn.

### Lỗi 4 — Đảo chữ số qua chuỗi nhưng giữ dạng chuỗi

In [26]:
def wrong_string_key(nums):
    cnt = Counter((str(x), str(x)[::-1]) for x in nums)   # SAI: khóa là cặp chuỗi, không phải hiệu
    return sum(v * (v - 1) // 2 for v in cnt.values()) % MOD

for nums, want in examples:
    print(nums, "sai:", wrong_string_key(nums), "| đúng:", want)

[42, 11, 1, 97] sai: 0 | đúng: 2
[13, 10, 35, 24, 76] sai: 0 | đúng: 4


**Vì sao sai:** khóa phải là **giá trị số** `x − rev(x)`; nhóm theo `(x, rev(x))` chỉ gom các phần tử **bằng nhau**, bỏ sót các cặp khác giá trị nhưng cùng hiệu (như `42` và `97`, cùng hiệu 18). **Sửa:** `x - int(str(x)[::-1])`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm cặp `(i, j)` thỏa `f(a_i, a_j) = g(a_i, a_j)`"** ⇒ thử **chuyển vế** để mỗi vế chỉ chứa một biến: `h(a_i) = h(a_j)` ⇒ đếm cặp cùng khóa.
- Đếm cặp cùng khóa: `Σ C(v, 2)` hoặc "cộng `seen[k]` rồi mới tăng".
- Biến thể "`h(a_i) + h(a_j) = T`" ⇒ two-sum: cộng `seen[T − h(a_j)]`.
- Đề có "modulo" ⇒ nhớ `%` ở kết quả.
- Hàm `rev` có số 0 cuối: cả cách số học lẫn `int(str[::-1])` đều tự xử lý.

## 15. Biến thể của bài

1. **LeetCode 1512 — Number of Good Pairs:** đếm cặp `nums[i] == nums[j]` (khóa chính là giá trị).
2. **LeetCode 2364 — Count Number of Bad Pairs:** cặp với `j − i ≠ nums[j] − nums[i]` ⇒ khóa `nums[i] − i`, lấy tổng cặp trừ cặp tốt.
3. **LeetCode 2001 — Number of Pairs of Interchangeable Rectangles:** khóa là tỉ số rút gọn `w/h`.
4. **Đếm cặp có `key` chênh nhau ≤ `K`** ⇒ sắp xếp khóa + hai con trỏ.
5. **Đếm bộ ba** cùng khóa: `Σ C(v, 3)` (Bài tập 5 dạng tổng quát hơn).

Minh họa biến thể **two-sum trên khóa**: đếm cặp `i < j` có `h(a_i) + h(a_j) = T` với `h(x) = x − rev(x)` (ví dụ tự dựng — đối chiếu bằng vét cạn):

In [27]:
def count_key_sum(nums, T):
    seen, ans = defaultdict(int), 0
    for x in nums:
        k = x - rev(x)
        ans += seen[T - k]            # tìm phần tử trước có khóa bù
        seen[k] += 1
    return ans

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(0, 120) for _ in range(rng.randint(1, 15))]
    T = rng.choice([0, 9, -9, 18, 27])
    n = len(nums)
    brute = sum(1 for i in range(n) for j in range(i + 1, n) if (nums[i] - rev(nums[i])) + (nums[j] - rev(nums[j])) == T)
    assert count_key_sum(nums, T) == brute
print("biến thể two-sum: OK; ví dụ T=0 trên [13, 31, 10, 1]:", count_key_sum([13, 31, 10, 1], 0))

biến thể two-sum: OK; ví dụ T=0 trên [13, 31, 10, 1]: 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | kiểm điều kiện gốc | `O(n²)` | `O(n)` | chuẩn đối chiếu |
| 2. Sắp xếp khóa | khúc bằng nhau | `O(n log n)` | `O(n)` | không muốn dùng băm; cần thứ tự khóa |
| 3. Counter + `C(v,2)` | tổ hợp theo nhóm | `O(n·d)` | `O(n)` | ngắn gọn |
| 4. Một lượt | `ans += seen[k]` | `O(n·d)` | `O(n)` | **nộp bài**, dễ mở rộng |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `countNicePairs([1, 2, 3])`, `([10, 20, 30])`, `([21, 12])`?

**Bài 2 (tính tay):** Tính khóa và đáp án cho `[352, 11, 253, 0, 22]`.

**Bài 3 (điền TODO):** Viết `list_nice_pairs(nums)` trả về **danh sách** mọi cặp đẹp `(i, j)` theo thứ tự tăng của `j` rồi `i`, trong `O(n + số cặp)`.

**Bài 4 (tự viết lại — LeetCode 2364):** Viết `count_bad_pairs(nums)`: số cặp `i < j` với `j − i ≠ nums[j] − nums[i]`.

**Bài 5 (mini challenge):** Đếm số **bộ ba** `i < j < k` mà ba phần tử đôi một tạo cặp đẹp, modulo `10⁹ + 7`.

In [28]:
def list_nice_pairs(nums):
    # TODO
    pass

def count_bad_pairs(nums):
    # TODO
    pass

def nice_triples(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Số một chữ số có `rev(x) = x` | `[1,2,3]` cùng khóa 0 | `10 − 1 = 9`, `20 − 2 = 18`…; `21 − 12 = 9`, `12 − 21 = −9` |
| 2 | `352 − 253 = 99`, `253 − 352 = −99` | `11`, `0`, `22` khóa 0 | Nhóm 0 có 3 phần tử ⇒ 3 cặp |
| 3 | `seen[k]` lưu **list chỉ số** | Ở bước `j` thêm `(i, j)` cho mỗi `i` trong list | Tổng thời gian tỉ lệ số cặp in ra |
| 4 | Cặp tốt ⇔ `nums[i] − i = nums[j] − j` | Tổng cặp `C(n, 2)` | Trừ số cặp tốt |
| 5 | Bộ ba đôi một đẹp ⇔ cùng khóa (tính bắc cầu của `=`) | Mỗi nhóm `v` góp `C(v, 3)` | `v(v−1)(v−2)/6` |

## 19. Đáp án tham khảo

In [29]:
r1 = [Solution().countNicePairs(a) for a in ([1, 2, 3], [10, 20, 30], [21, 12])]
print("Bài 1:", r1); assert r1 == [3, 0, 0]
print("Bài 2:"); r2 = trace_pairs([352, 11, 253, 0, 22]); assert r2 == 3 == nice_pairs_brute([352, 11, 253, 0, 22])

Bài 1: [3, 0, 0]
Bài 2:
j=0 x=352 rev=253 key=  99 | cặp mới [] | ans=0
j=1 x= 11 rev= 11 key=   0 | cặp mới [] | ans=0
j=2 x=253 rev=352 key= -99 | cặp mới [] | ans=0
j=3 x=  0 rev=  0 key=   0 | cặp mới [(1, 3)] | ans=1
j=4 x= 22 rev= 22 key=   0 | cặp mới [(1, 4), (3, 4)] | ans=3


In [30]:
def list_nice_pairs(nums):
    seen = defaultdict(list); out = []
    for j, x in enumerate(nums):
        k = x - rev(x)
        out.extend((i, j) for i in seen[k])
        seen[k].append(j)
    return out

def count_bad_pairs(nums):
    n = len(nums)
    seen, good = defaultdict(int), 0
    for i, x in enumerate(nums):
        good += seen[x - i]
        seen[x - i] += 1
    return n * (n - 1) // 2 - good

rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(0, 150) for _ in range(rng.randint(1, 15))]
    n = len(nums)
    brute = [(i, j) for j in range(n) for i in range(j) if nums[i] + rev(nums[j]) == nums[j] + rev(nums[i])]
    assert list_nice_pairs(nums) == brute
    assert count_bad_pairs(nums) == sum(1 for i in range(n) for j in range(i + 1, n) if j - i != nums[j] - nums[i])
assert count_bad_pairs([4, 1, 3, 3]) == 5 and count_bad_pairs([1, 2, 3, 4, 5]) == 0   # ví dụ của LeetCode 2364
print("Bài 3, 4: OK", list_nice_pairs([42, 11, 1, 97]))

Bài 3, 4: OK [(1, 2), (0, 3)]


In [31]:
def nice_triples(nums):
    cnt = Counter(x - rev(x) for x in nums)
    return sum(v * (v - 1) * (v - 2) // 6 for v in cnt.values()) % MOD

def nice_triples_brute(nums):
    n = len(nums); key = [x - rev(x) for x in nums]
    return sum(1 for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)
               if key[i] == key[j] == key[k]) % MOD

rng = random.Random(9)
for _ in range(300):
    nums = [rng.randint(0, 60) for _ in range(rng.randint(1, 14))]
    assert nice_triples(nums) == nice_triples_brute(nums)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", nice_triples([1, 2, 3, 11, 22]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 10


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
seen, ans = defaultdict(int), 0
for x in nums:
    k = x - int(str(x)[::-1])     # tách biến: a - rev(a) == b - rev(b)
    ans += seen[k]                # ghép với các phần tử TRƯỚC cùng khóa
    seen[k] += 1                  # tăng SAU khi cộng
return ans % (10**9 + 7)
```

**Checklist:**
- [ ] Khóa là **hiệu** `x − rev(x)` (không phải tổng)?
- [ ] `rev(120) = 21` (số 0 cuối biến mất)?
- [ ] Cộng `seen[k]` **trước** khi tăng (không đếm `(i, i)`)?
- [ ] Công thức nhóm `v(v−1)/2`?
- [ ] `% (10**9 + 7)` ở kết quả.